# Large Models Test Baseline

Closed-book API baseline for larger reference models on all 60 test problems. The default list is Qwen-family first, with DeepSeek left as an optional upper-bound comparison.


## Setup

This notebook currently uses OpenRouter's OpenAI-compatible API. Put `OPENROUTER_API_KEY=...` in the repo-local `.env` file.

In [ ]:
!pip install -q -U openai pandas tqdm python-dotenv

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

In [ ]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
PROJECT_ROOT

In [ ]:
from training_eval.eval_utils import (
    default_test_dir,
    extract_json_object,
    is_correct,
    load_jsonl_records,
    make_closed_book_prompt,
    rows_to_frame,
    save_results,
    summarize_accuracy,
)

## Load Dev Records

In [ ]:
DATA_DIR = default_test_dir(PROJECT_ROOT)
records = load_jsonl_records(DATA_DIR, pattern="*_preview.jsonl")
len(records), DATA_DIR

## Configure API Models

Now that the API has credit, the default references are larger Qwen-family models. Keep this list flexible: comment out anything slow, expensive, or rate-limited.


In [ ]:
client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

MODEL_NAMES = [
    "qwen/qwen3-8b",
    "qwen/qwen3-32b",
    "qwen/qwen3-235b-a22b",
    "deepseek/deepseek-v4-flash:free",
    "openai/gpt-oss-20b:free",
]

MAX_TOKENS = 128

RERUN_API_ERRORS = True
RERUN_EMPTY_OUTPUTS = True
FILL_ONLY_EXISTING_EMPTY_OUTPUTS = False
STOP_MODEL_ON_RATE_LIMIT = True

MAX_RETRY_PASSES = 100
SLEEP_BETWEEN_PASSES_SECONDS = 300


In [ ]:
def safe_model_name(model_name):
    return model_name.replace("-", "_").replace(".", "_").replace("/", "_").replace(":", "_")


def result_dir_for(model_name):
    return PROJECT_ROOT / "results" / "baselines" / f"{safe_model_name(model_name)}_dev_closed_book_api"


def outputs_path_for(model_name):
    return result_dir_for(model_name) / "outputs.jsonl"


def load_cached_rows(model_name):
    path = outputs_path_for(model_name)
    if not path.exists():
        return []
    rows_by_id = {}
    with path.open() as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                rows_by_id[row["id"]] = row
    return list(rows_by_id.values())


def append_cached_row(model_name, row):
    result_dir_for(model_name).mkdir(parents=True, exist_ok=True)
    with outputs_path_for(model_name).open("a") as f:
        f.write(json.dumps(row, sort_keys=True) + "\n")


def should_rerun_cached_row(row):
    if RERUN_API_ERRORS and row.get("api_error"):
        return True
    if RERUN_EMPTY_OUTPUTS and not str(row.get("raw_output", "")).strip():
        return True
    return False


def is_rate_limit_error(error_text):
    if not error_text:
        return False
    text = str(error_text).lower()
    return "429" in text or "rate limit" in text or "rate_limit" in text or "temporarily rate-limited" in text


def save_model_results(model_name, rows):
    df = rows_to_frame(rows)
    result_dir = result_dir_for(model_name)
    metrics = summarize_accuracy(df)
    metrics.update({
        "model": model_name,
        "provider": "openrouter",
        "dataset": "benchmark/data/test/*.jsonl",
        "num_api_errors": int(df["api_error"].notna().sum()) if "api_error" in df else 0,
        "num_empty_outputs": int(df["raw_output"].fillna("").astype(str).str.strip().eq("").sum()) if "raw_output" in df else 0,
        "checkpointed_after_each_record": True,
    })
    return save_results(rows, result_dir, metrics)


def generate_answer(problem, model_name):
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": make_closed_book_prompt(problem)}],
        temperature=0,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content or ""


## One-Example Smoke Test

This runs the first dev record and writes it into the normal `outputs.jsonl` cache. The full run will skip it instead of spending the call again.


In [ ]:
smoke_record = records[0]

for model_name in MODEL_NAMES:
    cached_rows = load_cached_rows(model_name)
    cached_by_id = {row["id"]: row for row in cached_rows if not should_rerun_cached_row(row)}

    print(model_name)
    if smoke_record["id"] in cached_by_id:
        row = cached_by_id[smoke_record["id"]]
        print("cached smoke result")
        print(row["raw_output"])
        print("predicted:", row["predicted_answer"])
        print("canonical:", row["canonical_answer"])
        print("correct:", row["correct"])
        print("api_error:", row.get("api_error"))
        print()
        continue

    metadata = smoke_record.get("metadata", {})
    raw_output = ""
    predicted = None
    api_error = None

    try:
        raw_output = generate_answer(smoke_record["problem"], model_name)
        if not raw_output.strip():
            api_error = "empty_response"
            correct = False
        else:
            predicted = extract_json_object(raw_output)
            correct = is_correct(predicted, smoke_record["canonical_answer"])
    except Exception as exc:
        api_error = repr(exc)
        correct = False

    row = {
        "id": smoke_record["id"],
        "family": smoke_record["family"],
        "problem_type": smoke_record["problem_type"],
        "difficulty": smoke_record["difficulty"],
        "manual_variation": metadata.get("manual_variation", False),
        "manual_problem_variation": metadata.get("manual_problem_variation", False),
        "manual_reasoning_variation": metadata.get("manual_reasoning_variation", False),
        "problem": smoke_record["problem"],
        "canonical_answer": smoke_record["canonical_answer"],
        "raw_output": raw_output,
        "predicted_answer": predicted,
        "api_error": api_error,
        "correct": correct,
    }

    append_cached_row(model_name, row)
    print(raw_output)
    print("predicted:", predicted)
    print("canonical:", smoke_record["canonical_answer"])
    print("correct:", correct)
    print("api_error:", api_error)
    print()


## Run Full Test Evaluation

This cell is safe to use with **Run All** and can be left running overnight.

Each pass skips good cached rows, retries cached empty/error rows, and runs missing rows. Each completed row is appended immediately to `outputs.jsonl`. If a model hits a rate limit, the current row is saved with `api_error`, that model pauses until the next pass, and the notebook continues with the next model.

If any rows are still unfinished at the end of a pass, the notebook sleeps for `SLEEP_BETWEEN_PASSES_SECONDS` before trying again.


In [ ]:
all_results = {}


def runnable_rows_for(model_name):
    cached_rows = load_cached_rows(model_name)
    good_rows = [row for row in cached_rows if not should_rerun_cached_row(row)]
    good_ids = {row["id"] for row in good_rows}
    unfinished = [record for record in records if record["id"] not in good_ids]
    return cached_rows, good_rows, good_ids, unfinished


for pass_idx in range(1, MAX_RETRY_PASSES + 1):
    print(f"\n=== Retry pass {pass_idx}/{MAX_RETRY_PASSES} ===")
    pass_started_rows = 0
    unfinished_after_pass = 0

    for model_name in MODEL_NAMES:
        cached_rows, rows, completed_ids, unfinished = runnable_rows_for(model_name)
        unfinished_after_pass += len(unfinished)

        if not unfinished:
            print(f"{model_name}: complete ({len(completed_ids)}/{len(records)})")
            all_results[model_name] = rows
            continue

        print(f"{model_name}: {len(completed_ids)}/{len(records)} good rows cached, {len(unfinished)} rows to fill/retry")
        model_hit_rate_limit = False

        for record in tqdm(unfinished):
            metadata = record.get("metadata", {})
            raw_output = ""
            predicted = None
            api_error = None

            try:
                raw_output = generate_answer(record["problem"], model_name)
                if not raw_output.strip():
                    api_error = "empty_response"
                    correct = False
                else:
                    predicted = extract_json_object(raw_output)
                    correct = is_correct(predicted, record["canonical_answer"])
            except Exception as exc:
                api_error = repr(exc)
                correct = False

            row = {
                "id": record["id"],
                "family": record["family"],
                "problem_type": record["problem_type"],
                "difficulty": record["difficulty"],
                "manual_variation": metadata.get("manual_variation", False),
                "manual_problem_variation": metadata.get("manual_problem_variation", False),
                "manual_reasoning_variation": metadata.get("manual_reasoning_variation", False),
                "problem": record["problem"],
                "canonical_answer": record["canonical_answer"],
                "raw_output": raw_output,
                "predicted_answer": predicted,
                "api_error": api_error,
                "correct": correct,
            }

            rows.append(row)
            append_cached_row(model_name, row)
            pass_started_rows += 1

            if STOP_MODEL_ON_RATE_LIMIT and is_rate_limit_error(api_error):
                print(f"Stopping {model_name} for this pass after rate-limit error on {record['id']}")
                model_hit_rate_limit = True
                break

        # Collapse duplicate cached rows and write metrics after every model attempt.
        rows = load_cached_rows(model_name)
        save_model_results(model_name, rows)
        all_results[model_name] = rows

    total_unfinished = 0
    for model_name in MODEL_NAMES:
        _, _, _, unfinished = runnable_rows_for(model_name)
        total_unfinished += len(unfinished)

    print(f"Pass {pass_idx} done. Rows attempted this pass: {pass_started_rows}. Remaining unfinished rows: {total_unfinished}.")

    if total_unfinished == 0:
        print("All model/question pairs are complete.")
        break

    print(f"Sleeping {SLEEP_BETWEEN_PASSES_SECONDS} seconds before next pass...")
    time.sleep(SLEEP_BETWEEN_PASSES_SECONDS)
else:
    print("Reached MAX_RETRY_PASSES before all rows completed.")


## Metrics

In [ ]:
{model: len(rows) for model, rows in all_results.items()} if "all_results" in globals() else {}


In [ ]:
for model_name, rows in all_results.items():
    df = rows_to_frame(rows)
    print(model_name)
    print(f"Overall accuracy: {df['correct'].mean():.3f} ({df['correct'].sum()}/{len(df)})")
    display(df.groupby("family")["correct"].agg(["mean", "sum", "count"]).sort_index())
    display(df.groupby("difficulty")["correct"].agg(["mean", "sum", "count"]).sort_index())
    display(df.groupby("manual_variation")["correct"].agg(["mean", "sum", "count"]).sort_index())
    print()

## Save Results

In [ ]:
saved_paths = {}

for model_name, rows in all_results.items():
    saved_paths[model_name] = save_model_results(model_name, rows)

saved_paths


## Inspect Mistakes

In [ ]:
available_models = [model_name for model_name, rows in all_results.items() if rows]
available_models


In [ ]:
model_name = available_models[-1]
df = rows_to_frame(all_results[model_name])
df.loc[~df["correct"], [
    "id",
    "family",
    "problem_type",
    "difficulty",
    "canonical_answer",
    "predicted_answer",
    "api_error",
    "raw_output",
]].head(20)
